# **Comparación de resultados entre datos originales y sintéticos**
## **Por:** Juan Esteban Álvarez García - 202212030

## **1 Preparación del ambiente de desarrollo**

In [1]:
!pip install pandas
!pip install numpy
!pip install matplotlib
!pip install scikit-learn
!pip install statsmodels
!pip install scipy
!pip install seaborn
!pip install spacy
!python -m spacy download es_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 99.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## **1.1 Imports y utilidades**

In [ ]:
# Imports esenciales
import re
import time
import nltk
import spacy
import string
import unicodedata
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from openai import OpenAI
from collections import Counter
from nltk.corpus import stopwords
from sklearn.utils import resample
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Opcionales
try:
    import statsmodels.api as sm
    import statsmodels.stats.api as sms
    from statsmodels.stats.outliers_influence import OLSInfluence
    STATS_MODELS_AVAILABLE = True
except Exception:
    STATS_MODELS_AVAILABLE = False

try:
    from scipy import stats
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

print("statsmodels:", STATS_MODELS_AVAILABLE, "| scipy:", SCIPY_AVAILABLE)

try:
    nlp = spacy.load("es_core_news_sm")
except OSError:
    print("El modelo 'es_core_news_sm' no se ha descargado. Por favor, ejecute el comando 'python -m spacy download es_core_news_sm'")
    exit()

#Helpers
# Descargar stopwords en español si no están disponibles
nltk.download('stopwords')

# Obtener stopwords en español
stop_words_es = set(stopwords.words('spanish'))
stop_words_es.add("él")              # U+00E9 (precompuesta)
stop_words_es.add("e\u0301l")        # U+0065 U+0301 (combinada)
stop_words_es.add("Él".lower())

# Definir las palabras adicionales que quieres eliminar
nuevas_stopwords = {'él', 'si', 'ejemplo', 'puede', 'pueden', 'ser'}
posibles_stopwords = {'países', 'sistema', 'años', 'datos', 'tasa', '1', '3', '4', 'ocde', 'cada'}

# Combinar las dos listas para crear la lista final de stopwords
stop_words_final = stop_words_es.union(nuevas_stopwords)
stop_words_final = stop_words_final.union(posibles_stopwords)

#def standardize(series):
#    s = pd.Series(series)
#    return (s - s.mean())/s.std(ddof=1)

#def ver_balance(df):
#    # Conteo
#    conteos = df['labels'].value_counts()
#    print("Conteos:", conteos)
#
#    # Proporción
#    porcentajes = df['labels'].value_counts(normalize=True) * 100
#    print("Porcentajes:", porcentajes.round(2))

def balancear_dataset_oversampling(df, columna_target='labels'):
    # Encontrar la clase con más muestras (para balancear hacia ella)
    distribucion_original = df[columna_target].value_counts()
    clase_referencia = distribucion_original.idxmax()
    tamaño_objetivo = distribucion_original.max()

    # Lista para almacenar DataFrames balanceados
    dfs_balanceados = []

    # Balancear cada clase
    for clase in df[columna_target].unique():
        df_clase = df[df[columna_target] == clase]

        if len(df_clase) < tamaño_objetivo:
            # Aplicar oversampling a clases minoritarias
            df_clase_balanced = resample(
                df_clase,
                replace=True,
                n_samples=tamaño_objetivo,
                random_state=42
            )
            dfs_balanceados.append(df_clase_balanced)
            print(f"Clase {clase}: {len(df_clase)} → {len(df_clase_balanced)}")
        else:
            # Mantener clases mayoritarias
            dfs_balanceados.append(df_clase)
            print(f"Clase {clase}: {len(df_clase)} (mantenida)")

    # Combinar y mezclar
    df_balanced = pd.concat(dfs_balanceados, ignore_index=True)
    df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

    return df_balanced

def preprocesar_texto(texto):
    if pd.isna(texto) or texto == '' or texto.isspace():
        return ''

    texto = str(texto)

    # Normalizar Unicode: convertir caracteres a su forma normalizada
    texto = unicodedata.normalize("NFC", texto)

    # Convertir a minúsculas
    texto = texto.lower()

    # Eliminar puntuación y caracteres especiales
    texto = texto.translate(str.maketrans('', '', string.punctuation))

    # Eliminar números
    texto = re.sub(r'\d+', '', texto)

    # Eliminar URLs
    texto = re.sub(r'http\S+', '', texto)

    # Eliminar menciones y hashtags (pero mantener el texto)
    texto = re.sub(r'@\w+', '', texto)
    texto = re.sub(r'#', '', texto)

    # Eliminar caracteres especiales pero mantener letras con acentos y espacios
    texto = re.sub(r'[^\wáéíóúñüàèìòùäëïöü\s]', ' ', texto)

    # Reemplazar múltiples espacios por uno solo
    texto = re.sub(r'\s+', ' ', texto)

    # Eliminar espacios al inicio y final
    texto = texto.strip()

    return texto

def contar_palabras(texto):
    if texto == '' or texto is None:
        return 0

    # Dividir el texto en palabras
    palabras = texto.split()

    # Filtrar palabras vacías
    palabras = [p for p in palabras if p != '']

    return len(palabras)

def eliminar_stopwords(texto):
    palabras = texto.split()
    palabras_filtradas = [p for p in palabras if p not in stop_words_final]

    return ' '.join(palabras_filtradas)

def lematizar_texto(texto):
    # Procesar el texto con spaCy
    doc = nlp(texto)
    # Obtener el lema de cada token y unirlos en una cadena
    lemas = [token.lemma_ for token in doc]
    return " ".join(lemas)

# Configurar API Key
client = OpenAI(api_key='REMPLAZAR EN COPIA LOCAL SIN SUBIR AL REPO')

def augmentar_dataset_con_openai(df, col_texto='textos', col_etiqueta='labels', objetivo_minimo=40):
    # Analizar distribución actual
    conteo_clases = Counter(df[col_etiqueta])
    print("Distribución actual:", dict(conteo_clases))

    textos_generados = []
    etiquetas_generadas = []

    for clase, cantidad in conteo_clases.items():
        if cantidad < objetivo_minimo:
            num_generar = objetivo_minimo - cantidad
            print(f"\nGenerando {num_generar} textos para clase {clase}...")

            # Obtener ejemplos reales de esta clase
            ejemplos_reales = df[df[col_etiqueta] == clase][col_texto].tolist()
            textos_muestra = "\n---\n".join(ejemplos_reales[:2])  # Usar máximo 3 ejemplos

            try:
                # Generar textos sintéticos
                textos_clase = generar_textos_openai(
                    textos_muestra,
                    clase,
                    num_generar
                )

                textos_generados.extend(textos_clase)
                etiquetas_generadas.extend([clase] * len(textos_clase))

                print(f"✓ Generados {len(textos_clase)} textos para clase {clase}")
                time.sleep(2)  # Rate limiting

            except Exception as e:
                print(f"✗ Error generando textos para clase {clase}: {e}")
                continue

    # Crear DataFrame con textos generados
    if textos_generados:
        df_generado = pd.DataFrame({
            col_texto: textos_generados,
            col_etiqueta: etiquetas_generadas
        })

        # Combinar con original
        df_balanceado = pd.concat([df, df_generado], ignore_index=True)
        print(f"\n✅ Dataset balanceado: {len(df)} → {len(df_balanceado)} textos")
        print("Nueva distribución:", dict(Counter(df_balanceado[col_etiqueta])))

        return df_balanceado
    else:
        print("⚠️ No se generaron nuevos textos")
        return df

def generar_textos_openai(textos_ejemplo, categoria, num_ejemplos):
    """
    Función que interactúa con OpenAI API
    """
    prompt = crear_prompt_balanceo(textos_ejemplo, categoria, num_ejemplos)

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            #model="gpt-4o",
            messages=[
                {"role": "system", "content": "Eres un experto en desarrollo sostenible y redacción técnica."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.85,
            max_tokens=4000,
            n=1
        )

        # Procesar respuesta con nueva sintaxis
        texto_respuesta = response.choices[0].message.content
        textos_generados = [texto.strip() for texto in texto_respuesta.split("===NUEVO_TEXTO===") if texto.strip()]

        return textos_generados[:num_ejemplos]

    except Exception as e:
        print(f"Error en API call: {e}")
        return []

def crear_prompt_balanceo(textos_ejemplo, categoria, num_ejemplos):
    ods_map = {
        1: "ODS 1 - Fin de la pobreza",
        3: "ODS 3 - Salud y bienestar",
        4: "ODS 4 - Educación de calidad"
    }

    prompt = f"""
Eres un experto en redacción de textos sobre Objetivos de Desarrollo Sostenible (ODS).
Tu tarea es generar {num_ejemplos} textos ORIGINALES y VARIADOS sobre {ods_map[categoria]}.

**CONTEXTO Y EJEMPLOS REALES:**
{textos_ejemplo}

**INSTRUCCIONES CRÍTICAS:**

1. **TEMÁTICA Y CONTENIDO:**
   - Enfócate ÚNICAMENTE en {ods_map[categoria]}
   - Los textos deben tener entre 1-2 párrafos coherentes
   - Varía perspectivas: políticas públicas, casos de estudio, análisis técnicos, testimonios ficticios pero realistas
   - Genera textos que parezcan escritos por diferentes autores/organizaciones
   - Usa vocabulario técnico y específico del ODS {ods_map[categoria]} de forma NATURAL
   - NO menciones explícitamente "ODS {categoria}" o "Objetivo de Desarrollo Sostenible {categoria}"

2. **PALABRAS CLAVE ESTRATÉGICAS** (incluir naturalmente, NO de forma forzada):
   - Para ODS 1: "pobreza", "pobre", "pobrezar", "privación", "hogar", "protección", "social", "ingreso", "ipm", "transferencia", "monetario", "matrimonio", "umbral", "banco", "pensión"
   - Para ODS 3: "salud", "médico", "atención", "sanitario", "paciente", "enfermedad", "mortalidad", "mental", "alcohol", "hospital", "droga", "tratamiento", "muerte", "medicamento", "hospitalario"
   - Para ODS 4: "educación", "escuela", "educativo", "estudiant", "alumno", "docente", "profesor", "aprendizaje", "escolar", "enseñanza", "maestro", "estudiante", "universidad", "pís", "ciencia"

3. **ESTILO Y CALIDAD:**
   - Varía el enfoque: puede ser narrativo, estadístico, testimonial, analítico o propositivo
   - Evitar repetición de frases exactas de los ejemplos
   - Coherencia lógica y gramatical
   - Diferentes enfoques regionales (Latinoamérica, África, Asia, etc.)

4. **FORMATO DE RESPUESTA:**
   - SOLO el texto generado, sin numeración
   - Separar cada texto completo con "===NUEVO_TEXTO==="

Genera {num_ejemplos} textos que sean realistas, útiles para entrenar modelos de ML y que capturen la esencia de {ods_map[categoria]}.
"""
    return prompt

def procesar_df_tagged(df_tagged):
    df_tagged['textos'] = df_tagged['textos'].astype(pd.StringDtype())
    print("Forma del dataset:", df_tagged.shape)
    display(df_tagged.dtypes)
    print("Valores nulos por columna:")
    print(df_tagged.isnull().sum())
    print("\nPorcentaje de nulos por columna:")
    print((df_tagged.isnull().sum() / len(df_tagged) * 100).round(2))
    filas_con_nulos = df_tagged[df_tagged.isnull().any(axis=1)]
    print(f"\nNúmero de filas con valores nulos: {len(filas_con_nulos)}")
    filas_con_texto_vacio = df_tagged[df_tagged['textos'] == '']
    print(f"\nNúmero de filas con texto vacío: {len(filas_con_texto_vacio)}")
    duplicados_totales = df_tagged.duplicated()
    df_tagged.dropna(inplace=True)
    print(f"Filas duplicadas exactas: {duplicados_totales.sum()}")
    duplicados_texto = df_tagged.duplicated(subset=['textos'])
    print(f"Filas con texto duplicado: {duplicados_texto.sum()}")
    df_tagged.drop_duplicates(inplace=True)
    df_tagged = balancear_dataset_oversampling(df_tagged) # Se balancean los datos
    df_tagged['textos'] = df_tagged['textos'].apply(preprocesar_texto)
    df_tagged['textos'] = df_tagged['textos'].apply(eliminar_stopwords)
    df_tagged['textos'] = df_tagged['textos'].apply(lematizar_texto)
    df_tagged['num_palabras'] = df_tagged['textos'].apply(contar_palabras)
    return df_tagged

def procesar_df_test(df_test):
    df_test['textos'] = df_test['textos'].astype(pd.StringDtype())
    print("Forma del dataset:", df_test.shape)
    display(df_test.dtypes)
    print("Valores nulos por columna:")
    print(df_test.isnull().sum())
    print("\nPorcentaje de nulos por columna:")
    print((df_test.isnull().sum() / len(df_test) * 100).round(2))
    filas_con_nulos = df_test[df_test.isnull().any(axis=1)]
    print(f"\nNúmero de filas con valores nulos: {len(filas_con_nulos)}")
    filas_con_texto_vacio = df_test[df_test['textos'] == '']
    print(f"\nNúmero de filas con texto vacío: {len(filas_con_texto_vacio)}")
    duplicados_totales = df_test.duplicated()
    df_test.dropna(inplace=True)
    print(f"Filas duplicadas exactas: {duplicados_totales.sum()}")
    duplicados_texto = df_test.duplicated(subset=['textos'])
    print(f"Filas con texto duplicado: {duplicados_texto.sum()}")
    df_test.drop_duplicates(inplace=True)
    df_test['textos'] = df_test['textos'].apply(preprocesar_texto)
    df_test['textos'] = df_test['textos'].apply(eliminar_stopwords)
    df_test['textos'] = df_test['textos'].apply(lematizar_texto)
    df_test['num_palabras'] = df_test['textos'].apply(contar_palabras)
    return df_test

statsmodels: True | scipy: True


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## **2 Aumentación de datos mediante prompting**
Se hace uso de las llaves de OpenAI para generar datos sintéticos a partir de los originales.

Se guarda el dataset aumentado para su revisión manual.

In [5]:
# Cargar el dataset para aumentar
df_raw = pd.read_excel('Datos_etapa 2.xlsx')

# Aumentar dataset
df_augmented = augmentar_dataset_con_openai(df_raw, objetivo_minimo=80)

# Guardar resultado
df_augmented.to_csv('dataset_augmentado.csv', index=False)
print("💾 Dataset augmentado guardado como 'dataset_augmentado.csv'")

Distribución actual: {1: 21, 3: 35, 4: 43}

Generando 59 textos para clase 1...
✓ Generados 40 textos para clase 1

Generando 45 textos para clase 3...
✓ Generados 31 textos para clase 3

Generando 37 textos para clase 4...
✓ Generados 25 textos para clase 4

✅ Dataset balanceado: 99 → 195 textos
Nueva distribución: {1: 61, 3: 66, 4: 68}
💾 Dataset augmentado guardado como 'dataset_augmentado.csv'


## **3 Procesamiento de los datos**
Se sigue la misma metodología de la Etapa 1 para el procesamiento de los datos:
 1. Eliminar nulos
 2. Eliminar duplicados
 3. Balanceo de clases
 4. Normalización de textos
    * Normalizar Unicode
    * Convertir a minúsculas
    * Eliminación de puntuaciones
    * Eliminación de caracteres especiales
    * Eliminación de números
    * Eliminación de URLs
    * Eliminación de Hashtags y Menciones
    * Normalización de espacios al interior y exterior de los textos
 5. Eliminación de StopWords
 6. Lematización de textos
 7. Conteo de palabras por texto
 8. Vectorización


In [8]:
# Procesar dataset original
print("Procesar dataset original")
df_original = procesar_df_tagged(pd.read_excel('Datos_etapa 2.xlsx')) # Pasos 1 al 7
vectorizer_original = TfidfVectorizer()
matrix_original = vectorizer_original.fit_transform(df_original['textos'])
print("Dimensiones de la matriz TF-IDF del conjunto original:", matrix_original.shape) # Paso 8

# Procesar dataset aumentado
print("\n\nProcesar dataset aumentado")
df_augmented = procesar_df_tagged(df_augmented) # Pasos 1 al 7
vectorizer_augmented = TfidfVectorizer()
matrix_augmented = vectorizer_augmented.fit_transform(df_augmented['textos'])
print("Dimensiones de la matriz TF-IDF del conjunto original:", matrix_augmented.shape) # Paso 8

Procesar dataset original
Forma del dataset: (99, 2)


,0
textos,string[python]
labels,int64


Valores nulos por columna:
textos    0
labels    0
dtype: int64

Porcentaje de nulos por columna:
textos    0.0
labels    0.0
dtype: float64

Número de filas con valores nulos: 0

Número de filas con texto vacío: 0
Filas duplicadas exactas: 0
Filas con texto duplicado: 0
Clase 1: 21 → 43
Clase 3: 35 → 43
Clase 4: 43 (mantenida)
Dimensiones de la matriz TF-IDF del conjunto original: (129, 1544)


Procesar dataset aumentado
Forma del dataset: (204, 3)


,0
textos,string[python]
labels,int64
num_palabras,int64


Valores nulos por columna:
textos          0
labels          0
num_palabras    0
dtype: int64

Porcentaje de nulos por columna:
textos          0.0
labels          0.0
num_palabras    0.0
dtype: float64

Número de filas con valores nulos: 0

Número de filas con texto vacío: 0
Filas duplicadas exactas: 52
Filas con texto duplicado: 52
Clase 1: 44 → 68
Clase 3: 40 → 68
Clase 4: 68 (mantenida)
Dimensiones de la matriz TF-IDF del conjunto original: (204, 1720)


## **4 Entrenamiento del modelo**


### **4.1 Separación de los datos de entrenamiento**

In [9]:
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

target = 'labels'

# Separación de los datos originales
X_train_original, X_val_original, y_train_original, y_val_original = train_test_split(
    matrix_original,
    df_original[target],
    test_size=0.2,
    random_state=42,
    stratify=df_original[target]
)

# Separación de los datos aumentados
X_train_augmented, X_val_augmented, y_train_augmented, y_val_augmented = train_test_split(
    matrix_augmented,
    df_augmented[target],
    test_size=0.2,
    random_state=42,
    stratify=df_augmented[target]
)

### **4.2 Búsqueda de hiperparámetros y métricas del modelo**
Se utiliza el modelo LinearSVC, que fue el que mejor rendimiento tuvo en la primera etapa

#### **4.2.1 Con los datos originales**

In [10]:
# Crear el modelo
svc_original = LinearSVC(dual=True)
param_grid = {
    "C": [0.01, 0.1, 0.25, 0.5, 1.0, 2.0, 4.0, 10.0, 100.0],
}
gs = GridSearchCV(
    estimator=svc_original,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1
)

# Buscar hiperparámetros
print("Buscando mejor C para LinearSVC...")
gs.fit(X_train_original, y_train_original)
best_svc_original = gs.best_estimator_
print(f"Mejor C: {gs.best_params_['C']}")

# Hacer predicciones
y_pred_original = best_svc_original.predict(X_val_original)

# Mostrar métricas
print("\n LinearSVC original")
print("Accuracy:", accuracy_score(y_val_original, y_pred_original))
print("F1-macro:", f1_score(y_val_original, y_pred_original, average="macro"))
print(classification_report(y_val_original, y_pred_original))

cm_original = confusion_matrix(y_val_original, y_pred_original)

labels =  ['ODS 1', 'ODS 3', 'ODS 4']

cm_df_original = pd.DataFrame(
    cm_original,
    index=[f"Verdadero {l}" for l in labels],
    columns=[f"Predicho {l}" for l in labels]
)

print("\nMatriz de Confusión en el conjunto original:")
print(cm_df_original)

Buscando mejor C para LinearSVC...
Mejor C: 0.1

 LinearSVC original
Accuracy: 0.8846153846153846
F1-macro: 0.8822055137844611
              precision    recall  f1-score   support

           1       1.00      0.75      0.86         8
           3       0.90      1.00      0.95         9
           4       0.80      0.89      0.84         9

    accuracy                           0.88        26
   macro avg       0.90      0.88      0.88        26
weighted avg       0.90      0.88      0.88        26


Matriz de Confusión en el conjunto original:
                 Predicho ODS 1  Predicho ODS 3  Predicho ODS 4
Verdadero ODS 1               6               0               2
Verdadero ODS 3               0               9               0
Verdadero ODS 4               0               1               8


#### **4.2.1 Con los datos aumentados**

In [11]:
# Crear el modelo
svc_augmented = LinearSVC(dual=True)
param_grid = {
    "C": [0.01, 0.1, 0.25, 0.5, 1.0, 2.0, 4.0, 10.0, 100.0],
}
gs = GridSearchCV(
    estimator=svc_augmented,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1
)

# Buscar hiperparámetros
print("Buscando mejor C para LinearSVC...")
gs.fit(X_train_augmented, y_train_augmented)
best_svc_augmented = gs.best_estimator_
print(f"Mejor C: {gs.best_params_['C']}")

# Hacer predicciones
y_pred_augmented = best_svc_augmented.predict(X_val_augmented)

# Mostrar métricas
print("\n LinearSVC augmented")
print("Accuracy:", accuracy_score(y_val_augmented, y_pred_augmented))
print("F1-macro:", f1_score(y_val_augmented, y_pred_augmented, average="macro"))
print(classification_report(y_val_augmented, y_pred_augmented))

cm_augmented = confusion_matrix(y_val_augmented, y_pred_augmented)

labels =  ['ODS 1', 'ODS 3', 'ODS 4']

cm_df_augmented = pd.DataFrame(
    cm_augmented,
    index=[f"Verdadero {l}" for l in labels],
    columns=[f"Predicho {l}" for l in labels]
)

print("\nMatriz de Confusión en el conjunto augmented:")
print(cm_df_augmented)

Buscando mejor C para LinearSVC...
Mejor C: 0.1

 LinearSVC augmented
Accuracy: 1.0
F1-macro: 1.0
              precision    recall  f1-score   support

           1       1.00      1.00      1.00        13
           3       1.00      1.00      1.00        14
           4       1.00      1.00      1.00        14

    accuracy                           1.00        41
   macro avg       1.00      1.00      1.00        41
weighted avg       1.00      1.00      1.00        41


Matriz de Confusión en el conjunto augmented:
                 Predicho ODS 1  Predicho ODS 3  Predicho ODS 4
Verdadero ODS 1              13               0               0
Verdadero ODS 3               0              14               0
Verdadero ODS 4               0               0              14


## **5 Comparación de resultados**
Se puede observar que las métricas del modelo que fue entrenado con los datos aumentados son mejores que las del modelo que fue entrenado con los datos originales.

El modelo que usó los datos aumentados acertó en todas sus predicciones, lo que le otorgó métricas perfectas de 1.0 tanto para precisión, recall, f1-score y accuracy.

Mientras que el modelo que usó los datos originales falló en 3 predicciones y sus métricas fueron 0.90 en precisión y 0.88 en las demás.

Esto puede significar que el modelo que usó los datos aumentados realmente logró aprender de ellos y hacer predicciones correctas, o que el modelo está incurriendo en over-fitting y no podrá hacer predicciones correctas cuando vea datos nuevos.

## **6 Prueba del modelo**
En este caso no hay escenarios de prueba sin etiquetar, así que se usa el dataset original y se comparan los labels reales con las predicciones del modelo entrenado con datos aumentados, para determinar si realmente logró aprender de los datos o si incurrió en overfitting

In [ ]:
def predict_ods(archivo_nuevos_datos, modelo, vectorizer):
    # Cargar datos
    if archivo_nuevos_datos.endswith('.xlsx'):
        df = pd.read_excel(archivo_nuevos_datos)
    else:
        df = pd.read_csv(archivo_nuevos_datos)

    # Asumir que la columna de texto se llama 'textos'
    if 'textos' not in df.columns:
        # Si no existe, usar la primera columna
        df.rename(columns={df.columns[0]: 'textos'}, inplace=True)

    # Guardar orden original
    df_resultado = df.copy()

    # Hacer una copia para procesar manteniendo el índice original
    df_para_procesar = df.copy()

    # Preprocesar (igual que en entrenamiento)
    df_clean = procesar_df_test(df_para_procesar)

    # Verificar que hay datos después del preprocesamiento
    if len(df_clean) == 0:
        print("⚠️  No hay textos válidos después del preprocesamiento")
        df_resultado['predicts'] = -1
        df_resultado['ODS'] = "NO_PREDECIDO"
        return df_resultado

    # Vectorizar y predecir
    X_new = vectorizer.transform(df_clean['textos'])
    predictions = modelo.predict(X_new)

    print(f"✅ Realizadas {len(predictions)} predicciones")
    print(f"Distribución de predicciones: {Counter(predictions)}")

    # Crear un mapeo usando el índice original en lugar del texto
    # Esto es más robusto porque no depende de que el texto coincida exactamente
    mapeo_predicciones = dict(zip(df_clean.index, predictions))

    # Inicializar la columna predicts con -1 (no predecido)
    df_resultado['predicts'] = -1
    df_resultado['ODS'] = "NO_PREDECIDO"

    # Asignar las predicciones a las filas correspondientes
    for idx, pred in mapeo_predicciones.items():
        if idx in df_resultado.index:
            df_resultado.loc[idx, 'predicts'] = pred

    # Mapear nombres ODS
    nombre_ods = {1: "ODS 1", 3: "ODS 3", 4: "ODS 4"}
    df_resultado['ODS'] = df_resultado['predicts'].map(nombre_ods)

    # Para los que siguen en -1, mantener "NO_PREDECIDO"
    df_resultado.loc[df_resultado['predicts'] == -1, 'ODS'] = "NO_PREDECIDO"

    return df_resultado

# Predicciones para datos sin etiquetar
df_predicciones = predict_ods("Datos_etapa 2.xlsx", best_svc_augmented, vectorizer_augmented)

df_predicciones.to_csv("predicciones.csv", index=False)

Forma del dataset: (99, 2)


,0
textos,string[python]
labels,int64


Valores nulos por columna:
textos    0
labels    0
dtype: int64

Porcentaje de nulos por columna:
textos    0.0
labels    0.0
dtype: float64

Número de filas con valores nulos: 0

Número de filas con texto vacío: 0
Filas duplicadas exactas: 0
Filas con texto duplicado: 0
✅ Realizadas 99 predicciones
Distribución de predicciones: Counter({np.int64(4): 50, np.int64(3): 33, np.int64(1): 16})


### **6.1 Resultados**
Luego de verificar manualmente los resultados, se obersvó que solo 7 de las 100 predicciones no coincidieron con los labels originales.

Se concluye que el modelo se entrenó correctamente, aprendió de los datos y está en capacidad de hacer predicciones valiosas para el negocio con datos nuevos sin etiquetar.